# WLS 04. Generate Tableau AI Insight Cards

Creates two Tableau-ready card tables from all weighted-correlation and WLS evidence.

- Insight: deterministic analysis confidence plus AI core summary and implication
- Driver: rule-based common and differentiated drivers

The pipeline creates analysis confidence and common/differentiated drivers first. The LLM then references those fixed results to generate the core summary and implication. Unchanged successful scopes reuse their existing AI card. Failed calls are recorded separately and retried on the next execution.

The card tables contain only scalar text and values, never JSON.

In [ ]:
import importlib
import sys

from pyspark.sql import functions as F

PROJECT_ROOT = '/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc'
SRC_ROOT = f'{PROJECT_ROOT}/src'
if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import driver.driver_dashboard_cards as dashboard_cards

importlib.reload(config_loader)
importlib.reload(dashboard_cards)

from common.config_loader import get_output_table, load_config
from driver.driver_dashboard_cards import generate_dashboard_cards

config = load_config(f'{PROJECT_ROOT}/config/settings_intellytics.yaml')


In [ ]:
# Optional selective run. Keep all values as None to generate every available scope.
TARGET_GROUP_DIMS = None  # e.g. ['brand_name']
TARGET_Y_FEATURES = None  # e.g. ['Remote Control Usability']
TARGET_GROUP_KEYS = None  # e.g. ['LG', 'Samsung', 'TCL']
MAX_PROFILES = None       # e.g. 10 for a low-cost smoke test

result = generate_dashboard_cards(
    spark,
    config,
    target_group_dims=TARGET_GROUP_DIMS,
    target_y_features=TARGET_Y_FEATURES,
    target_group_keys=TARGET_GROUP_KEYS,
    max_profiles=MAX_PROFILES,
)
result


In [ ]:
for table_key in [
    'driver_card_insight',
    'driver_card_driver',
    'driver_card_generation_log',
]:
    table_name = get_output_table(config, table_key)
    print(f'\n[{table_key}] {table_name}')
    if spark.catalog.tableExists(table_name):
        display(spark.table(table_name).orderBy('group_dim', 'y_feature', 'group_key').limit(30))
    else:
        print('No table created yet: check profile_count and llm_failed_insight_count in the previous cell.')
